In [4]:
from sim import *
import subprocess

REQUESTS_FILE = "data/requests/ann-arbor.parquet"

In [14]:
FLEET_SIZES = (7500, 10000, 12500, 15000)

pax2 = 0.0
pax4 = 0.0

args = ["./sim"]
for charging in ("in-place", "closest-hub"):
    for fleet_size in FLEET_SIZES:
        suffix = f"{fleet_size}-{charging}-{pax2}-{pax4}"

        with open(f"data/sim_configs/ann-arbor-{suffix}.txt", "w") as f:
            f.write(f"length: {24 * 7 * 3600}\n")
            f.write(f"requests: {REQUESTS_FILE}\n")
            f.write(f"riders: data/riders/ann-arbor-{fleet_size}.parquet\n")
            f.write(f"output: output/raw/ann-arbor-{suffix}\n")

            f.write(f"frac_2_seater: {pax2}\n")
            f.write(f"frac_4_seater: {pax4}\n")

            f.write(f"strategy: bounded-h3\n")
            f.write(f"charging: {charging}\n")

        args.append(f"ann-arbor-{suffix}")

subprocess.run(["cmake", "--build", "cmake-build-release", "--target", "sim", "-j", "8"], check=True)
subprocess.run(args, check=True)

[ 11%] Built target tqdm
[ 47%] Built target h3
[ 52%] Built target pandas
[100%] Built target sim
...launching 4 threads
reading data/requests/ann-arbor.parquet
reading data/requests/ann-arbor.parquet
reading reading data/requests/ann-arbor.parquet
data/requests/ann-arbor.parquet
reading data/riders/ann-arbor-7500.parquet
reading data/riders/ann-arbor-15000.parquet
reading data/riders/ann-arbor-10000.parquet
reading data/riders/ann-arbor-12500.parquet


[##################################################] 100% | [90s< 0s]]
[##################################################] 100% | [131s< 0s]
[##################################################] 100% | [147s< 0s]
[##################################################] 100% | [160s< 0s]


simulation complete
...saving fleet statistics (count=677)
...saving rider statistics (count=12525366)
...saving statistics (count=3346447)
simulation complete
...saving fleet statistics (count=677)
...saving rider statistics (count=13561248)
...saving statistics (count=3346447)
simulation complete
...saving fleet statistics (count=676)
...saving rider statistics (count=13559462)
...saving statistics (count=3346447)
simulation complete
...saving fleet statistics (count=677)
...saving rider statistics (count=13557662)
...saving statistics (count=3346447)
Elapsed time: 178982 milliseconds



CompletedProcess(args=['./sim', 'ann-arbor-7500-closest-hub-0.0-0.0', 'ann-arbor-10000-closest-hub-0.0-0.0', 'ann-arbor-12500-closest-hub-0.0-0.0', 'ann-arbor-15000-closest-hub-0.0-0.0'], returncode=0)

In [15]:
requests = pd.read_parquet(REQUESTS_FILE)

for charging in ("in-place", "closest-hub"):
    for fleet_size in FLEET_SIZES:
        suffix = f"{fleet_size}-{charging}-{pax2}-{pax4}"

        all_output, output, fleet_output, waypoints_output, rider_states = (
            load_results(requests, f"output/raw/ann-arbor-{suffix}"))

        stats = compute_stats(all_output, output)
        rider_stats = compute_rider_stats(waypoints_output, rider_states)

        print()
        print(charging, fleet_size)
        print(f"Service Level: {stats['service_level']:.2f}%")
        print(f"Mean FM Distance: {stats['fm_dist']:.3f} km")
        print(f"Mean Response Time: {stats['response_time']:.2f} mins")
        print(f"% Spent Charging: {rider_stats['charge_pct']:.2f}%")
        print(f"% Spent Idle: {rider_stats['idle_pct']:.2f}%")


closest-hub 7500
Service Level: 92.21%
Mean FM Distance: 1.573 km
Mean Response Time: 7.22 mins
% Spent Charging: 8.09%
% Spent Idle: 41.90%

closest-hub 10000
Service Level: 100.00%
Mean FM Distance: 0.943 km
Mean Response Time: 2.75 mins
% Spent Charging: 5.93%
% Spent Idle: 56.36%

closest-hub 12500
Service Level: 100.00%
Mean FM Distance: 0.902 km
Mean Response Time: 2.37 mins
% Spent Charging: 4.77%
% Spent Idle: 64.94%

closest-hub 15000
Service Level: 100.00%
Mean FM Distance: 0.861 km
Mean Response Time: 2.09 mins
% Spent Charging: 4.01%
% Spent Idle: 70.56%
